# Notebook extract language resolution

**Status:** design, user-approved to specify and implement  
**Beads epic:** `bd-2c1c`  
**Plan id:** `nb-extract-lang-2026-08-24`  
**Date:** 2026-08-24  
**Approved approach:** explicit `code_type` never falls through to the kernel; MCP tokens bind to extract `Language`; `ns_mermaid` is Mermaid (not Markdown, not skip); `.ipynb` stays exclusive of JSON; cell identity remains `path#cell:{id}`.

## Solver evidence (pre-spec)

Catalog families `configuration` and `data_integrity` (Z3 4.16.0):

| Query | Status | solve_id |
|---|---|---|
| `.ipynb` → Jupyter, `.json` → JSON, unique path | `sat` / pass | `sol_cd8077487c144b3e` |
| Unique stable key with `#cell:{id}` | `sat` / pass | `sol_676c631a5b044131` |
| Unique on file+offset only (pre-`eb92d8806` bug) | `unsat` / fail | `sol_f9ad2331c27146f5` |
| Current `ns_mermaid` + python kernel → python | `unsat` / fail | `sol_cf8d6904f9514ee4` |
| Target `ns_mermaid` → mermaid bind | `sat` / pass | `sol_353ba8defa3b4000` |
| MCP tokens vs extract bindings (current) | `unsat` / fail | `sol_c23f6d1896e74eef` |
| Target ns_mermaid requires Mermaid, excludes Python/Markdown | `sat` / pass | `sol_d4457d1860ca4c54` |
| crates.io `tree-sitter-mermaid` 0.1.0 vs rust 1.88 / tree-sitter 0.25 | `unsat` / fail | `sol_c8ac66e7d9e3481e` |

Same-cell NS-Mermaid gates below re-prove the implementation partitions (`@spec TOKEN-RESOLVE`, `TOKEN-MAP`, `CONTAINER-OWNER`).

## Problem

`extract_notebook_file` is a JSON container dispatcher. Per-cell language uses `resolve_cell_language`, which `find_map`s the fallback chain. A **present** but unmapped `metadata.spur.code_type` (`ns_mermaid`, `go`, `sql`) falls through to `kernelspec` (usually `python3`) and is parsed as the wrong grammar. SPUR design notebooks are polyglot: python kernel + `ns_mermaid` cells.

JSON extract (`1ff8d61eb`) correctly kept `.ipynb` exclusive. Cell stable keys (`eb92d8806`) correctly use `path#cell:{id}`. Those must stay.

## Goals

1. If `code_type` is present: map it or skip. Never fall through to kernel/language_info.
2. Map MCP tokens `python|javascript|rust|go|sql|ns_mermaid` (aliases `python3`, `evcxr`, `gonb`, `mermaid`).
3. `ns_mermaid` / `mermaid` → `Language::Mermaid` with a tree-sitter **0.25** / rust **≤1.88** grammar.
4. Keep `.ipynb` as Jupyter container (empty tags, specialized dispatch). Optional `.mmd` / `.mermaid` file routing to Mermaid.
5. Keep `#cell:{id}` identity keys. Prefer notebook `cell.id`; if absent, a content-stable fallback — not a moving `cell-{idx}` as the only key.

## Non-goals

- Parsing `.ipynb` with the JSON extractor.
- crates.io `tree-sitter-mermaid` 0.1.0 (rust 1.95, tree-sitter 0.26, ~322k C lines).
- Julia / R grammars (no registry language; skip is correct).
- New `NodeKind` for diagrams.
- Rewriting the markdown-span File node for the container (empty queries stay).

## Gate 1 — explicit `code_type` never falls through

`resolve_cell_language` today: `candidates.flatten().find_map(language_for_token)`. That skips an unmapped present token and tries the kernel.

Contract:

| `code_type` present? | Token maps? | Action |
|---|---|---|
| yes | yes | use that language |
| yes | no | **skip** (warn) |
| no | n/a | kernel / `language_info` fallback chain |

The kernel is legal only when no explicit `code_type` exists. Formal unit: `@spec TOKEN-RESOLVE`.

In [ ]:
flowchart TD
    SPEC["`@spec TOKEN-RESOLVE
@type Action = enum[use_explicit, skip, use_kernel]
@input has_explicit: Bool
@input mapped: Bool
@output status: Action
@requires PRE: true`"]
    EXPLICIT["`@branch EXPLICIT
@when has_explicit = true and mapped = true
@ensures USE_EXPLICIT: status = use_explicit`"]
    SKIP["`@branch SKIP
@when has_explicit = true and mapped = false
@ensures SKIP_STATUS: status = skip`"]
    KERNEL["`@branch KERNEL
@when has_explicit = false
@ensures USE_KERNEL: status = use_kernel`"]
    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCLUSIVE: prove partition_exclusive
@verify STATUSES: witness each status`"]
    SPEC --> EXPLICIT --> CHECK
    SPEC --> SKIP --> CHECK
    SPEC --> KERNEL --> CHECK

## Gate 2 — MCP token → extract language

`language_for_token` is the closed map. MCP `code_type` enum is `python|javascript|rust|go|sql|ns_mermaid`. Aliases (`python3`, `evcxr`, `gonb`, `mermaid`) bind to the same languages. `julia` (and any other unmapped token) skips. `ns_mermaid` is **Mermaid**, never Markdown.

Go and SQL already have `Language::{Go,Sql}`. Mermaid needs a new `Language::Mermaid` + grammar + `queries/mermaid/tags.scm`.

Formal unit: `@spec TOKEN-MAP`.

In [ ]:
flowchart TD
    SPEC["`@spec TOKEN-MAP
@type Token = enum[python, python3, javascript, rust, evcxr, go, gonb, sql, ns_mermaid, mermaid, julia]
@type Lang = enum[python, javascript, rust, go, sql, mermaid, skip]
@input token: Token
@output status: Lang
@requires PRE: true`"]
    PY["`@branch PY
@when token = python or token = python3
@ensures PY_LANG: status = python`"]
    JS["`@branch JS
@when token = javascript
@ensures JS_LANG: status = javascript`"]
    RS["`@branch RS
@when token = rust or token = evcxr
@ensures RS_LANG: status = rust`"]
    GO["`@branch GO
@when token = go or token = gonb
@ensures GO_LANG: status = go`"]
    SQL["`@branch SQL
@when token = sql
@ensures SQL_LANG: status = sql`"]
    MM["`@branch MM
@when token = ns_mermaid or token = mermaid
@ensures MM_LANG: status = mermaid`"]
    SKIP["`@branch SKIP
@when token = julia
@ensures SKIP_LANG: status = skip`"]
    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCLUSIVE: prove partition_exclusive
@verify STATUSES: witness each status`"]
    SPEC --> PY --> CHECK
    SPEC --> JS --> CHECK
    SPEC --> RS --> CHECK
    SPEC --> GO --> CHECK
    SPEC --> SQL --> CHECK
    SPEC --> MM --> CHECK
    SPEC --> SKIP --> CHECK

## Gate 3 — container vs file languages

`.ipynb` remains the Jupyter **container** (specialized dispatch, empty tags). JSON extract must not claim it. Mermaid source files (`.mmd`, `.mermaid`) route to `Language::Mermaid` — they are not notebooks.

JupyterNotebook keeps `tree_sitter_md` + empty queries only for the File span. Do not run JSON or Markdown tags on the container bytes.

Formal unit: `@spec CONTAINER-OWNER`.

In [ ]:
flowchart TD
    SPEC["`@spec CONTAINER-OWNER
@type Path = enum[ipynb, json, mmd, mermaid_ext]
@type Owner = enum[jupyter, json, mermaid]
@input path: Path
@output status: Owner
@requires PRE: true`"]
    NB["`@branch NB
@when path = ipynb
@ensures NB_OWNER: status = jupyter`"]
    JSON["`@branch JSON
@when path = json
@ensures JSON_OWNER: status = json`"]
    MM["`@branch MM
@when path = mmd or path = mermaid_ext
@ensures MM_OWNER: status = mermaid`"]
    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCLUSIVE: prove partition_exclusive
@verify STATUSES: witness each status`"]
    SPEC --> NB --> CHECK
    SPEC --> JSON --> CHECK
    SPEC --> MM --> CHECK

## Architecture

```
.ipynb bytes
  -> Language::JupyterNotebook (matcher .ipynb only)
  -> serde_json parse (not tree-sitter JSON)
  -> File node (markdown grammar, empty queries, whole-file span)
  -> for each cell:
       emit Cell node + Contains
       markdown cell -> Language::Markdown
       code cell -> TOKEN-RESOLVE then TOKEN-MAP
         mapped -> parse cell source with that grammar + tags.scm
                   identity_path = {file}#cell:{stable_id}
         skip -> warn, no symbols
```

`stable_id` = notebook `cell.id` if present, else a content hash of cell source (not a shifting `cell-{idx}` as the sole key).

## Parser issues this spec owns

| Issue | Disposition |
|---|---|
| Explicit `code_type` falls through to kernel (`find_map`) | **fix** — TOKEN-RESOLVE |
| `ns_mermaid` / `mermaid` unmapped | **fix** — TOKEN-MAP + `Language::Mermaid` |
| `go` / `gonb` / `sql` unmapped despite existing grammars | **fix** — TOKEN-MAP only |
| Same local offset colliding across cells | **already fixed** (`eb92d8806`); keep `#cell:{id}` |
| `cell-{idx}` moves on insert | **fix** — content-stable fallback |
| JSON stealing `.ipynb` | **already guarded** (`1ff8d61eb`); keep CONTAINER-OWNER |
| Container uses markdown grammar | **keep** empty queries; do not add tags |
| `extract_cell` too_many_arguments | **refactor** args into a struct while touching the call |
| crates.io `tree-sitter-mermaid` 0.1.0 | **reject** — wrap a 0.25 / rust ≤1.88 grammar (monaqa `parser.c` + `tree-sitter-language`, not the 0.20 crate) |

## Mermaid adapter (v1)

Same registry contract as JSON: `Language::Mermaid` + matcher `.mmd`/`.mermaid` + `queries/mermaid/tags.scm`.

- Flowchart / graph vertex ids and subgraph ids → `Module`
- Sequence participants → `Module`
- No new `NodeKind`; relations `{contains, defines}` only
- `builtin_method_names` → `&[]`
- NS-Mermaid typed annotations are still Mermaid source; extract named carriers, do not run the NS solver in spur-graph

## Tests (TDD)

1. **RED:** `code_type: ns_mermaid` + root `kernelspec: python3` does **not** resolve to Python (skip until Mermaid lands, then Mermaid).
2. **RED:** `code_type: go` / `sql` resolve to `Language::Go` / `Sql` even with a python kernel.
3. **RED:** `code_type: julia` + python kernel still skips (does not become Python).
4. **RED:** two mermaid cells with the same local vertex offset get distinct stable keys.
5. **RED:** mermaid flowchart fixture emits Module symbols for vertex ids.
6. **Keep:** `jupyter_keeps_ipynb_and_does_not_share_json`, `same_cell_local_symbol_offsets_get_distinct_stable_keys`.

## Acceptance

- Formal cells `TOKEN-RESOLVE`, `TOKEN-MAP`, `CONTAINER-OWNER` run with matching solver statuses.
- `language_for_token` matches TOKEN-MAP; `resolve_cell_language` matches TOKEN-RESOLVE.
- Polyglot notebook (python + js + rust + go + sql + ns_mermaid) extracts each cell with the mapped grammar.
- `.ipynb` is never JSON; `.mmd` is Mermaid.
- Workspace rust-version stays 1.88; tree-sitter stays 0.25.